# JPX 特征分析

1. 特征重要性排序
2. 特征间相关性
3. 消融实验
4. 特征与Target的关联

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_merged
from src.data.preprocessor import preprocess
from src.features.build_features import build_all_features, get_feature_columns
from src.models.train_lgb import train_lightgbm

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
%matplotlib inline

In [ ]:
df = load_merged(use_train=True)
df = preprocess(df)
df = build_all_features(df, use_cache=True)
feature_cols = get_feature_columns(df)
print(f'特征数量: {len(feature_cols)}')
print(f'特征列表: {feature_cols}')

## 1. 训练模型获取特征重要性

In [ ]:
model, importance = train_lightgbm(df, feature_cols=feature_cols)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
top20 = importance.head(20)
ax.barh(range(len(top20)), top20['importance'].values)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20['feature'].values)
ax.set_xlabel('重要性 (Gain)')
ax.set_title('特征重要性 Top 20')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 2. 特征间相关性

In [ ]:
top_features = importance.head(15)['feature'].tolist()
corr_matrix = df[top_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Top 15 特征相关性矩阵')
plt.tight_layout()
plt.show()

## 3. 特征与 Target 的相关系数

In [ ]:
target_corr = df[feature_cols + ['Target']].corr()['Target'].drop('Target').sort_values()
fig, ax = plt.subplots(figsize=(10, 8))
target_corr.plot.barh(ax=ax)
ax.set_title('各特征与 Target 的 Pearson 相关系数')
ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()